In [ ]:
# D4 (docs/SCALE_PLAN.md): SFT LoRA del 4B campeon sobre el snapshot ARC-Heavy (19K, dataset
# privado arcscale-archeavy-19k). Receta = la validada en el carril proxy (D1/D1-bis/D2-bis):
# LoRA r=64 alpha 32 all-proj, lr 1e-4, accum 8, secuencias completas <=2048 tok (filtro con
# el tokenizer real, no truncado). Eval integrada: 24 tareas NUEVAS de AGI-2 (no-AGI-1, la
# poblacion limpia de D1-bis) antes y despues. Salida: adapter + modelo FUSIONADO (bf16) para
# el kernel de inferencia. Offline (L4x4, 1 GPU); imagen pinneada (gate de peft/torchao).
import json, os, random, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

T00 = time.time()
N_TRAIN = 8000   # dimensionado por D2 (ajustar si la curva 2000->8000 sigue subiendo)
MAX_LEN = 2048

# --- inputs (Kaggle monta via symlink -> followlinks) ---
MDIR, JSONL = None, None
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if MDIR is None and "config.json" in files and any(x.endswith(".safetensors") for x in files):
        MDIR = root
    if JSONL is None and "arc_heavy_19k.jsonl" in files:
        JSONL = os.path.join(root, "arc_heavy_19k.jsonl")
assert MDIR and JSONL, (MDIR, JSONL)
print(f"[setup] modelo={MDIR}\n[setup] data={JSONL}", flush=True)

TOK = AutoTokenizer.from_pretrained(MDIR)

def g2s(grid):
    return "\n".join("".join(str(int(c)) for c in row) for row in grid)

def fmt_pair(inp, out):
    return f"<|im_start|>user\n{g2s(inp)}<|im_end|><|im_start|>assistant\n{g2s(out)}<|im_end|>"

def fmt_query(inp):
    return f"<|im_start|>user\n{g2s(inp)}<|im_end|><|im_start|>assistant\n"

# --- items de entrenamiento (mismo formato del proxy: contexto = pares menos el ultimo) ---
TRAIN = []
with open(JSONL) as fh:
    for line in fh:
        pairs = json.loads(line)["pairs"]
        prompt = "".join(fmt_pair(i, o) for i, o in pairs[:-1]) + fmt_query(pairs[-1][0])
        ans = g2s(pairs[-1][1]) + "<|im_end|>"
        pi = TOK(prompt, return_tensors="pt").input_ids[0]
        ai = TOK(ans, return_tensors="pt").input_ids[0]
        if pi.numel() + ai.numel() > MAX_LEN:
            continue
        TRAIN.append(torch.cat([pi, ai]))
        if len(TRAIN) >= N_TRAIN:
            break
random.Random(7).shuffle(TRAIN)
print(f"[setup] {len(TRAIN)} items de entrenamiento ({time.time()-T00:.0f}s)", flush=True)

# --- eval: las 24 tareas NUEVAS de AGI-2 del holdout (poblacion limpia de D1-bis) ---
NEW24 = ["6bcdb01e", "9caf5b84", "84551f4c", "5ad8a7c0", "b2bc3ffd", "25e02866", "84ba50d3",
         "e4941b18", "4df5b0ae", "230f2e48", "d6e50e54", "5623160b", "c61be7dc", "d749d46f",
         "c803e39c", "18286ef8", "80214e03", "fc4aaf52", "c6141b15", "c920a713", "48634b99",
         "22208ba4", "e84fef15", "ad173014"]
CDIR = "/kaggle/input/arc-prize-2026-arc-agi-2"
CHAL = json.load(open(f"{CDIR}/arc-agi_training_challenges.json"))
SOLS = json.load(open(f"{CDIR}/arc-agi_training_solutions.json"))
ITEMS = {}
for k in NEW24:
    t = CHAL[k]
    prompt = "".join(fmt_pair(p["input"], p["output"]) for p in t["train"]) + fmt_query(t["test"][0]["input"])
    pi = TOK(prompt, return_tensors="pt").input_ids[0]
    if pi.numel() > 6000:
        continue
    ai = TOK(g2s(SOLS[k][0]) + "<|im_end|>", return_tensors="pt").input_ids[0]
    ITEMS[k] = (pi, ai)
print(f"[setup] {len(ITEMS)} tareas de eval", flush=True)

# --- modelo (bf16 en la L4; sin GradScaler) + LoRA + grad checkpointing ---
MODEL = AutoModelForCausalLM.from_pretrained(MDIR, torch_dtype=torch.bfloat16, device_map="cuda:0")
LCFG = LoraConfig(r=64, lora_alpha=32, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"])
MODEL = get_peft_model(MODEL, LCFG)
MODEL.enable_input_require_grads()
MODEL.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
MODEL.config.use_cache = False
print(f"[setup] 4B + LoRA r=64 | mem={torch.cuda.memory_allocated()/1e9:.1f}GB "
      f"({time.time()-T00:.0f}s)", flush=True)

@torch.no_grad()
def nll_ids(pi, ai):
    ids = torch.cat([pi, ai]).unsqueeze(0).to("cuda:0")
    logits = MODEL(ids).logits.float()
    lp = torch.log_softmax(logits[0, pi.numel() - 1:-1], -1)
    return -lp.gather(1, ai.unsqueeze(1).to("cuda:0")).mean().item()

def greedy(key, max_new=960):
    pi, _ = ITEMS[key]
    ids = pi.unsqueeze(0).to("cuda:0")
    MODEL.config.use_cache = True
    with torch.no_grad():
        out = MODEL.generate(ids, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=TOK.eos_token_id)
    MODEL.config.use_cache = False
    txt = TOK.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    try:
        pred = [[int(c) for c in line if c.isdigit()] for line in txt.strip().split("\n") if line.strip()]
        gt = SOLS[key][0]
        if not pred or len(pred) != len(gt) or any(len(r) != len(gt[0]) for r in pred):
            return 0.0, False
        p, g = np.array(pred), np.array(gt)
        return float((p == g).mean()), bool((p == g).all())
    except Exception:
        return 0.0, False

def eval_point(tag):
    MODEL.eval()
    torch.cuda.empty_cache()
    rows = []
    for k in ITEMS:
        pi, ai = ITEMS[k]
        nll = nll_ids(pi, ai)
        acc, exact = greedy(k)
        rows.append({"task": k, "nll": round(nll, 4), "cell_acc": round(acc, 4), "exact": exact})
    summ = {"tag": tag, "nll_media": round(float(np.mean([r["nll"] for r in rows])), 4),
            "cell_acc_media": round(float(np.mean([r["cell_acc"] for r in rows])), 4),
            "exactas": sum(r["exact"] for r in rows), "n": len(rows)}
    print("RESULT_POINT:" + json.dumps(summ), flush=True)
    return {"resumen": summ, "detalle": rows}

RES = {"points": [eval_point("base_4b")]}

ACCUM, LR = 8, 1e-4
OPT = torch.optim.AdamW([p for p in MODEL.parameters() if p.requires_grad], lr=LR)
MODEL.config.use_cache = False
MODEL.train()
t0 = time.time()
OPT.zero_grad()
for n, full in enumerate(TRAIN):
    ids = full.unsqueeze(0).to("cuda:0")
    out = MODEL(ids, labels=ids)
    (out.loss / ACCUM).backward()
    if (n + 1) % ACCUM == 0:
        OPT.step()
        OPT.zero_grad()
    if (n + 1) % 500 == 0:
        print(f"[train] {n+1}/{len(TRAIN)} loss={out.loss.item():.3f} "
              f"({time.time()-t0:.0f}s, pico {torch.cuda.max_memory_allocated()/1e9:.1f}GB)",
              flush=True)
OPT.step()
OPT.zero_grad()
MODEL.eval()
print(f"[train] completo en {time.time()-t0:.0f}s", flush=True)

RES["points"].append(eval_point(f"sft_n{len(TRAIN)}"))
json.dump(RES, open("/kaggle/working/scale_sft_result.json", "w"), indent=1)

# --- salidas: adapter (resume/backup) + modelo FUSIONADO (para el kernel de inferencia) ---
MODEL.save_pretrained("/kaggle/working/adapter_4b")
MERGED = MODEL.merge_and_unload()
MERGED.save_pretrained("/kaggle/working/merged_4b", safe_serialization=True)
TOK.save_pretrained("/kaggle/working/merged_4b")
# Leccion del 0.00 (12-ago, ensayo 2B): un config.json escrito por otro transformers puede
# perder rope_theta y romper el modelo en el stack del kernel de inferencia. Se envian los
# archivos auxiliares ORIGINALES del 4B (el merge no cambia arquitectura ni tokenizer).
import shutil
for f in ("config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json",
          "special_tokens_map.json", "added_tokens.json", "vocab.json", "merges.txt"):
    src = os.path.join(MDIR, f)
    if os.path.exists(src):
        shutil.copy(src, "/kaggle/working/merged_4b/")
print("[out] adapter_4b + merged_4b guardados (aux files originales)", flush=True)